# Computer Vision and Deep Learning — Laboratory 4
## Semantic Segmentation with U-Net

This notebook implements per-pixel semantic segmentation on the **Labeled Faces in the Wild (LFW)** parts dataset.
Each pixel is assigned one of three classes:

| Label | Class | Colour in visualisations |
|-------|-------|--------------------------|
| 0 | background | black |
| 1 | skin / face | red |
| 2 | hair | blue |

### Outline

1. [Dataset](#1-dataset) — `LFWDataset`, download helpers, 80/10/10 split
2. [Augmentation](#2-augmentation) — hand-written paired transforms + `torchvision.transforms.v2`
3. [U-Net Architecture](#3-u-net-architecture) — Encoder, bottleneck, Decoder, skip connections
4. [Loss Functions](#4-loss-functions) — Cross-entropy, Dice, and a combined loss
5. [Evaluation Metrics](#5-evaluation-metrics) — pixel accuracy, IoU, mIoU, freq-weighted IoU
6. [Training](#6-training) — training loop, cosine-LR schedule, wandb logging + overlay images
7. [Post-processing](#7-post-processing) — confidence thresholding, probability smoothing, blob removal
8. [Experiments](#8-experiments) — upsampling strategy · loss comparison · post-processing impact

In [18]:
%matplotlib inline

In [19]:
import os
import hashlib
import tarfile
import logging
import subprocess
import time
import random

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import requests
from tqdm import tqdm
import cv2
import scipy.ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from torchvision.tv_tensors import Mask
from PIL import Image

import wandb

In [20]:
def _default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

DEVICE = _default_device()
NUM_CLASSES = 3
print(f'Device: {DEVICE}')

Device: mps


## 1. Dataset

The [LFW parts dataset](https://vis-www.cs.umass.edu/lfw/) extends the standard Labeled Faces in the Wild image collection with per-pixel segmentation masks.
Not every image has a corresponding mask — `LFWDataset` builds the index by scanning the mask directory and only keeping pairs where the `.jpg` image also exists.

**Split** (80 / 10 / 10 train / val / test) is derived from a fixed random shuffle (seed 42).

Raw PPM mask pixel values are normalised to contiguous class indices `{0, 1, 2}` in `__getitem__`.

### Manual download (if the UMass server is unreachable)

The official host `vis-www.cs.umass.edu` is intermittently unavailable. If the automated download fails, download the two archives manually and extract them into `./lfw_dataset/`:

| File | Source |
|------|--------|
| `lfw_funneled/` (images) | [Kaggle — lfw-dataset](https://www.kaggle.com/datasets/jessicali9530/lfw-dataset) |
| `parts_lfw_funneled_gt_images/` (masks) | [Kaggle — lfw-part-labels](https://www.kaggle.com/datasets/arunmohan003/lfw-part-labels) |

Then re-run the cell below with `download=False`.

In [21]:
_MANUAL_DOWNLOAD_HELP = """
Cannot reach vis-www.cs.umass.edu (the official LFW host is currently unavailable).
Please download the two archives manually and place them in {base_folder}/:

  1. LFW funneled images:
       https://www.kaggle.com/datasets/jessicali9530/lfw-dataset
       → rename / place as:  {base_folder}/lfw-funneled.tgz
       → OR extract directly so that {base_folder}/lfw_funneled/ exists

  2. LFW parts segmentation masks:
       https://www.kaggle.com/datasets/arunmohan003/lfw-part-labels
       → rename / place as:  {base_folder}/parts_lfw_funneled_gt_images.tgz
       → OR extract directly so that {base_folder}/parts_lfw_funneled_gt_images/ exists

Then re-run this cell with download=False.
"""


class LFWDataset(torch.utils.data.Dataset):
    """
    LFW segmentation dataset (3 classes: 0=background, 1=skin, 2=hair).
    Only image/mask pairs where both files exist are included.
    """
    NUM_CLASSES = 3
    _DATA = (
        # images
        ('http://vis-www.cs.umass.edu/lfw/lfw-funneled.tgz', None),
        # segmentation masks as ppm
        ('https://vis-www.cs.umass.edu/lfw/part_labels/parts_lfw_funneled_gt_images.tgz',
         '3e7e26e801c3081d651c8c2ef3c45cfc'),
    )
    # Expected extracted directory names (used to skip download when already present)
    _DIRS = ('lfw_funneled', 'parts_lfw_funneled_gt_images')

    def __init__(self, base_folder, transforms=None, download=True,
                 split_name='train', img_size=128):
        super().__init__()
        self.base_folder = base_folder
        self.transforms  = transforms
        self.img_size    = img_size

        if download:
            self.download_resources(base_folder)

        mask_dir = os.path.join(base_folder, 'parts_lfw_funneled_gt_images')
        img_dir  = os.path.join(base_folder, 'lfw_funneled')

        pairs = []
        for person in sorted(os.listdir(mask_dir)):
            person_mask_dir = os.path.join(mask_dir, person)
            if not os.path.isdir(person_mask_dir):
                continue
            for mf in sorted(os.listdir(person_mask_dir)):
                if not mf.endswith('.ppm'):
                    continue
                stem     = os.path.splitext(mf)[0]
                img_path = os.path.join(img_dir, person, stem + '.jpg')
                msk_path = os.path.join(person_mask_dir, mf)
                if os.path.exists(img_path):
                    pairs.append((img_path, msk_path))

        rng = random.Random(42)
        rng.shuffle(pairs)
        n       = len(pairs)
        n_train = int(0.8 * n)
        n_val   = int(0.9 * n)
        splits  = {
            'train': pairs[:n_train],
            'val':   pairs[n_train:n_val],
            'test':  pairs[n_val:],
        }
        self.X = [p[0] for p in splits[split_name]]
        self.Y = [p[1] for p in splits[split_name]]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img     = Image.open(self.X[idx]).convert('RGB')
        raw     = np.array(Image.open(self.Y[idx]))
        if raw.ndim == 3:
            raw = raw[:, :, 0]
        # Normalize raw pixel values to class indices 0..NUM_CLASSES-1
        mx = int(raw.max())
        if mx > self.NUM_CLASSES - 1:
            raw = np.round(raw.astype(float) / mx * (self.NUM_CLASSES - 1)).astype(np.int64)
        else:
            raw = raw.astype(np.int64)
        mask_pil = Image.fromarray(raw.astype(np.uint8))

        if self.transforms is not None:
            img_t, mask_t = self.transforms(img, mask_pil)
        else:
            img      = img.resize((self.img_size, self.img_size), Image.BILINEAR)
            mask_pil = mask_pil.resize((self.img_size, self.img_size), Image.NEAREST)
            img_t    = TF.to_image(img)
            img_t    = TF.to_dtype(img_t, dtype=torch.float32, scale=True)
            mask_t   = torch.from_numpy(np.array(mask_pil))

        if not isinstance(mask_t, torch.Tensor):
            mask_t = torch.from_numpy(np.array(mask_t))
        return img_t, mask_t.long()

    # ------------------------------------------------------------------ download helpers

    def download_resources(self, base_folder):
        os.makedirs(base_folder, exist_ok=True)
        if all(os.path.isdir(os.path.join(base_folder, d)) for d in self._DIRS):
            return
        try:
            # masks first, then images — matches PDF order
            self._download_and_extract_archive(
                url=LFWDataset._DATA[1][0], base_folder=base_folder, md5=LFWDataset._DATA[1][1])
            self._download_and_extract_archive(
                url=LFWDataset._DATA[0][0], base_folder=base_folder, md5=None)
        except Exception as e:
            raise RuntimeError(
                _MANUAL_DOWNLOAD_HELP.format(base_folder=base_folder)
            ) from e

    def _download_and_extract_archive(self, url, base_folder, md5) -> None:
        base_folder = os.path.expanduser(base_folder)
        filename    = os.path.basename(url)
        archive     = os.path.join(base_folder, filename)
        marker      = archive + '.extracted'
        if os.path.exists(marker):
            return
        self._download_url(url, base_folder, md5)
        print(f'Extracting {archive} to {base_folder}')
        self._extract_tar_archive(archive, base_folder, True)
        open(marker, 'w').close()

    def _retreive(self, url, save_location, chunk_size: int = 1024 * 32) -> None:
        try:
            response   = requests.get(url, stream=True)
            total_size = int(response.headers.get('content-length', 0))
            with open(save_location, 'wb') as file, tqdm(
                desc=os.path.basename(save_location),
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
            ) as bar:
                for data in response.iter_content(chunk_size=chunk_size):
                    file.write(data)
                    bar.update(len(data))
            print(f'Download successful. File saved to: {save_location}')
        except Exception as e:
            print(f'An error occurred: {str(e)}')
            raise

    def _download_url(self, url: str, base_folder: str, md5: str = None) -> None:
        base_folder = os.path.expanduser(base_folder)
        filename    = os.path.basename(url)
        file_path   = os.path.join(base_folder, filename)
        os.makedirs(base_folder, exist_ok=True)
        if self._check_file(file_path, md5):
            print(f'File {file_path} already exists. Using that version')
            return
        print(f'Downloading {url} to file_path')
        self._retreive(url, file_path)
        if not self._check_file(file_path, md5):
            raise RuntimeError('File not found or corrupted.')

    def _extract_tar_archive(self, from_path: str, to_path: str = None,
                             remove_finished: bool = False) -> str:
        if to_path is None:
            to_path = os.path.dirname(from_path)
        with tarfile.open(from_path, 'r') as tar:
            tar.extractall(to_path)
        if remove_finished:
            os.remove(from_path)
        return to_path

    def _compute_md5(self, filepath: str, chunk_size: int = 1024 * 1024) -> str:
        with open(filepath, 'rb') as f:
            md5 = hashlib.md5()
            while chunk := f.read(chunk_size):
                md5.update(chunk)
        return md5.hexdigest()

    def _check_file(self, filepath: str, md5: str) -> bool:
        if not os.path.isfile(filepath):
            return False
        if md5 is None:
            return True
        return self._compute_md5(filepath) == md5

In [22]:
BASE_FOLDER = './lfw_dataset'
IMG_SIZE    = 128

# Set download=True only if the dataset has not been placed manually yet.
# If vis-www.cs.umass.edu is unreachable, the class prints instructions for
# manual download (Kaggle links) and raises a RuntimeError.
train_ds = LFWDataset(BASE_FOLDER, split_name='train', img_size=IMG_SIZE, download=True)
val_ds   = LFWDataset(BASE_FOLDER, split_name='val',   img_size=IMG_SIZE, download=False)
test_ds  = LFWDataset(BASE_FOLDER, split_name='test',  img_size=IMG_SIZE, download=False)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

An error occurred: HTTPSConnectionPool(host='vis-www.cs.umass.edu', port=443): Max retries exceeded with url: /lfw/part_labels/parts_lfw_funneled_gt_images.tgz (Caused by NameResolutionError("HTTPSConnection(host='vis-www.cs.umass.edu', port=443): Failed to resolve 'vis-www.cs.umass.edu' ([Errno 8] nodename nor servname provided, or not known)"))


RuntimeError: 
Cannot reach vis-www.cs.umass.edu (the official LFW host is currently unavailable).
Please download the two archives manually and place them in ./lfw_dataset/:

  1. LFW funneled images:
       https://www.kaggle.com/datasets/jessicali9530/lfw-dataset
       → rename / place as:  ./lfw_dataset/lfw-funneled.tgz
       → OR extract directly so that ./lfw_dataset/lfw_funneled/ exists

  2. LFW parts segmentation masks:
       https://www.kaggle.com/datasets/arunmohan003/lfw-part-labels
       → rename / place as:  ./lfw_dataset/parts_lfw_funneled_gt_images.tgz
       → OR extract directly so that ./lfw_dataset/parts_lfw_funneled_gt_images/ exists

Then re-run this cell with download=False.


In [ ]:
bs         = 4
dataloader = torch.utils.data.DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=0)

for i_batch, sample_batched in enumerate(dataloader):
    imgs = sample_batched[0]
    segs = sample_batched[1]
    print(i_batch, imgs.size(), segs.size())

    rows, cols = bs, 2
    figure = plt.figure(figsize=(bs * 2, rows))
    for i in range(bs):
        figure.add_subplot(rows, cols, 2 * i + 1)
        plt.title('image')
        plt.axis('off')
        plt.imshow(imgs[i].permute(1, 2, 0).numpy().clip(0, 1))
        figure.add_subplot(rows, cols, 2 * i + 2)
        plt.title('seg')
        plt.axis('off')
        plt.imshow(segs[i].numpy(), cmap='gray')
    plt.show()

    if i_batch == 2:
        break

## 2. Augmentation

Segmentation augmentation must apply **identical geometric transforms** to both the image and its mask, otherwise the pixel-level correspondence breaks.

### 2a. Hand-written paired geometric augmentations

`PairedCompose` sequences transforms that each accept and return `(image, mask)` pairs:
- `ResizePair` — bilinear for images, nearest-neighbour for masks (preserves class indices)
- `RandomHorizontalFlipPair` — applies the same coin flip to both
- `ToTensorPair` — converts PIL images to float32 / int64 tensors
- `NormalizePair` — normalises the image tensor; passes mask through unchanged

In [ ]:
class ResizePair:
    """Resize image (bilinear) and mask (nearest-neighbour) to the same size."""
    def __init__(self, size):
        self.size = size  # (H, W)

    def __call__(self, img: Image.Image, mask: Image.Image):
        img  = img.resize((self.size[1], self.size[0]), Image.BILINEAR)
        mask = mask.resize((self.size[1], self.size[0]), Image.NEAREST)
        return img, mask


class RandomHorizontalFlipPair:
    """Applies the same random horizontal flip to both image and mask."""
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, img: Image.Image, mask: Image.Image):
        if random.random() < self.p:
            img  = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        return img, mask


class ToTensorPair:
    """Convert (PIL image, PIL mask) → (float32 tensor, int64 tensor)."""
    def __call__(self, img: Image.Image, mask: Image.Image):
        img_t  = TF.to_image(img)
        img_t  = TF.to_dtype(img_t, dtype=torch.float32, scale=True)
        mask_t = torch.from_numpy(np.array(mask).astype(np.int64))
        return img_t, mask_t


class NormalizePair:
    """Normalize image tensor; pass mask through unchanged."""
    def __init__(self, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)):
        self._norm = T.Normalize(mean, std)

    def __call__(self, img_t: torch.Tensor, mask_t: torch.Tensor):
        return self._norm(img_t), mask_t


class PairedCompose:
    """Sequentially applies a list of paired (img, mask) transforms."""
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, img, mask):
        for t in self.transforms:
            img, mask = t(img, mask)
        return img, mask


def make_train_transforms(img_size=128):
    return PairedCompose([
        ResizePair((img_size, img_size)),
        RandomHorizontalFlipPair(p=0.5),
        ToTensorPair(),
        NormalizePair(),
    ])


def make_val_transforms(img_size=128):
    return PairedCompose([
        ResizePair((img_size, img_size)),
        ToTensorPair(),
        NormalizePair(),
    ])

### 2b. Paired transforms using `torchvision.transforms.v2`

In v2, wrapping the mask as `tv_tensors.Mask` makes the transform pipeline type-aware:
geometric ops apply identically to both; `Normalize` is skipped for masks; `Resize` uses nearest interpolation for masks automatically.

In [ ]:
def make_v2_train_transforms(img_size=128):
    """Paired train transforms via torchvision.transforms.v2."""
    geometric = T.Compose([
        T.Resize((img_size, img_size)),       # NEAREST for Mask, BILINEAR for Image
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10),
    ])
    image_only = T.Compose([
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

    def apply(img_pil: Image.Image, mask_pil: Image.Image):
        img_t  = TF.to_image(img_pil)                                    # (C,H,W) uint8
        mask_t = Mask(torch.from_numpy(np.array(mask_pil).astype(np.int64)))
        img_t, mask_t = geometric(img_t, mask_t)                         # same random state
        img_t = image_only(img_t)
        return img_t, mask_t.long()

    return apply

In [ ]:
# Verify that flip is applied identically to image and mask
sample_img, sample_mask = train_ds[0]
img_pil  = Image.fromarray((sample_img.permute(1, 2, 0).numpy().clip(0, 1) * 255).astype(np.uint8))
mask_pil = Image.fromarray(sample_mask.numpy().astype(np.uint8))

flip = RandomHorizontalFlipPair(p=1.0)
flipped_img, flipped_mask = flip(img_pil, mask_pil)

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for ax, im, title in zip(axes.flat,
                         [img_pil, flipped_img, mask_pil, flipped_mask],
                         ['Original Image', 'Flipped Image', 'Original Mask', 'Flipped Mask']):
    ax.imshow(im, cmap='gray' if 'Mask' in title else None)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. U-Net Architecture

U-Net (Ronneberger et al., 2015) is an encoder–decoder network with **skip connections** that short-circuit spatial detail from the encoder directly into the decoder.

```
Input (B, 3, H, W)
  │
  ▼  ConvBlock × N  +  MaxPool × N  ──── skips[0..N-1] ────┐
  │                                                          │
  ▼  Bottleneck ConvBlock                                    │
  │                                                          │
  ▼  DecoderBlock × N:                                       │
       Upsample  ──  CenterCrop skip  ──  concat  ──  ConvBlock
  │
  ▼  1×1 Conv → C logits
  │
  ▼  F.interpolate → (B, C, H, W)
```

**`use_bilinear`** switches the upsample step between `ConvTranspose2d` (learnable) and bilinear `Upsample` + 1×1 conv (smoother, fewer parameters) — tested in Experiment 1.

In [ ]:
class ConvBlock(nn.Module):
    """Two 3×3 convolutions, each followed by BatchNorm + ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Encoder(nn.Module):
    """
    U-Net encoder.  channel_list = [in_ch, c1, c2, ...]
    forward returns (x_after_last_pool, skips) where skips[i] is the
    pre-pool output of block i (used as skip connections).
    """
    def __init__(self, channel_list):
        super().__init__()
        self.blocks = nn.ModuleList()
        self.pools  = nn.ModuleList()
        for i in range(len(channel_list) - 1):
            self.blocks.append(ConvBlock(channel_list[i], channel_list[i + 1]))
            self.pools.append(nn.MaxPool2d(2, 2))

    def forward(self, x):
        skips = []
        for block, pool in zip(self.blocks, self.pools):
            x = block(x)
            skips.append(x)       # save before pooling
            x = pool(x)
        return x, skips

In [ ]:
def upsample_block(x, filters, size, stride=2):
    """
    x       - the input of the upsample block
    filters - the number of filters to be applied
    size    - the size of the filters
    """
    in_channels = x.shape[1]
    block = nn.Sequential(
        nn.ConvTranspose2d(in_channels, filters, size, stride=stride),
        nn.BatchNorm2d(filters),
        nn.ReLU(inplace=True),
    ).to(x.device)
    return block(x)


in_layer    = torch.rand((32, 32, 128, 128))
filter_sz   = 4
num_filters = 16
for stride in [2, 4, 8]:
    x = upsample_block(in_layer, num_filters, filter_sz, stride)
    print('in shape: ', in_layer.shape, ' upsample with filter size ', filter_sz,
          '; stride ', stride, ' -> out shape ', x.shape)

In [ ]:
class DecoderBlock(nn.Module):
    """
    Upsample → CenterCrop skip → concat → ConvBlock.
    use_bilinear=True replaces ConvTranspose2d with bilinear Upsample + 1×1 conv.
    """
    def __init__(self, in_ch, skip_ch, out_ch, use_bilinear=False):
        super().__init__()
        if use_bilinear:
            self.up = nn.Sequential(
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
            )
        else:
            self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        # CenterCrop skip to match x's spatial size
        h, w = x.shape[2], x.shape[3]
        dh   = (skip.shape[2] - h) // 2
        dw   = (skip.shape[3] - w) // 2
        skip = skip[:, :, dh:dh + h, dw:dw + w]
        return self.conv(torch.cat([x, skip], dim=1))


class Decoder(nn.Module):
    """
    U-Net decoder.
    skip_channels: encoder channel sizes ordered deepest-first, e.g. [512, 256, 128, 64].
    """
    def __init__(self, bottleneck_ch, skip_channels, use_bilinear=False):
        super().__init__()
        self.blocks = nn.ModuleList()
        in_ch = bottleneck_ch
        for skip_ch in skip_channels:
            out_ch = skip_ch
            self.blocks.append(DecoderBlock(in_ch, skip_ch, out_ch, use_bilinear))
            in_ch = out_ch
        self.out_channels = in_ch

    def forward(self, x, skips):
        """x: bottleneck tensor; skips: list ordered deepest-first."""
        for block, skip in zip(self.blocks, skips):
            x = block(x, skip)
        return x


class UNet(nn.Module):
    def __init__(self, in_channels=3, enc_channels=(64, 128, 256, 512),
                 num_classes=3, use_bilinear=False):
        super().__init__()
        enc_list        = [in_channels] + list(enc_channels)
        self.encoder    = Encoder(enc_list)
        bottleneck_out  = enc_channels[-1] * 2
        self.bottleneck = ConvBlock(enc_channels[-1], bottleneck_out)
        skip_channels   = list(reversed(enc_channels))   # deepest first
        self.decoder    = Decoder(bottleneck_out, skip_channels, use_bilinear)
        self.head       = nn.Conv2d(self.decoder.out_channels, num_classes, 1)

    def forward(self, x):
        h_in, w_in    = x.shape[2], x.shape[3]
        x_enc, skips  = self.encoder(x)
        x_bot         = self.bottleneck(x_enc)
        x_dec         = self.decoder(x_bot, list(reversed(skips)))
        logits        = self.head(x_dec)
        return F.interpolate(logits, size=(h_in, w_in), mode='bilinear', align_corners=False)

In [ ]:
# Sanity check
_model = UNet(in_channels=3, enc_channels=(32, 64, 128, 256), num_classes=NUM_CLASSES)
_dummy = torch.rand(2, 3, 128, 128)
_out   = _model(_dummy)
assert _out.shape == (2, NUM_CLASSES, 128, 128), f'Unexpected shape: {_out.shape}'
n_params = sum(p.numel() for p in _model.parameters())
print(f'Input: {tuple(_dummy.shape)}  ->  Output: {tuple(_out.shape)}')
print(f'Parameters: {n_params:,}')
del _model, _dummy, _out

## 4. Loss Functions

| Loss | Formula | Notes |
|------|---------|-------|
| Cross-entropy | \(-\sum_c y_c \log \hat{p}_c\) | Strong gradient signal; sensitive to class imbalance |
| Dice | \(1 - \frac{2\sum p \cdot y + \varepsilon}{\sum p + \sum y + \varepsilon}\) | Directly optimises overlap; more robust to imbalance |
| Combined | \(\mathcal{L}_{CE} + \lambda \cdot \mathcal{L}_{Dice}\) | Benefits of both; λ = 1 by default |

The Dice loss is implemented in *soft* form: predictions are softmax probabilities rather than hard class indices, making it fully differentiable. All three variants are compared in **Experiment 2**.

In [ ]:
def dice_loss(logits, targets, num_classes=NUM_CLASSES, smooth=1.0):
    """Soft multi-class Dice loss (vectorised, no explicit pixel loops)."""
    probs      = F.softmax(logits, dim=1)                                   # (B, C, H, W)
    targets_oh = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
    p_flat     = probs.reshape(probs.shape[0], num_classes, -1)             # (B, C, N)
    t_flat     = targets_oh.reshape(targets_oh.shape[0], num_classes, -1)
    inter      = (p_flat * t_flat).sum(dim=2)                               # (B, C)
    denom      = p_flat.sum(dim=2) + t_flat.sum(dim=2)
    dice       = (2.0 * inter + smooth) / (denom + smooth)
    return 1.0 - dice.mean()


def combined_loss(logits, targets, num_classes=NUM_CLASSES, lam=1.0):
    """CE + λ·Dice."""
    return F.cross_entropy(logits, targets) + lam * dice_loss(logits, targets, num_classes)

## 5. Evaluation Metrics

All four metrics are **fully vectorised** — no explicit Python loops over pixels.
The confusion matrix is built in a single `torch.bincount` call, from which TP / FP / FN are read off as row/column sums.

| Metric | Definition |
|--------|-----------|
| Pixel accuracy | \(\frac{\text{correct pixels}}{\text{total pixels}}\) |
| IoU per class | \(\frac{TP_c}{TP_c + FP_c + FN_c}\) |
| Mean IoU (mIoU) | \(\frac{1}{C} \sum_c \text{IoU}_c\) |
| Frequency-weighted IoU | \(\sum_c \frac{n_c}{N} \cdot \text{IoU}_c\) |

In [ ]:
@torch.no_grad()
def pixel_accuracy(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Fraction of correctly classified pixels."""
    return (pred == target).float().mean().item()


@torch.no_grad()
def iou_per_class(pred: torch.Tensor, target: torch.Tensor,
                  num_classes: int) -> torch.Tensor:
    """
    Per-class IoU via confusion matrix (vectorised).
    Returns a tensor of shape (num_classes,).
    """
    flat_pred   = pred.reshape(-1)
    flat_target = target.reshape(-1)
    valid       = (flat_target >= 0) & (flat_target < num_classes)
    flat_pred   = flat_pred[valid]
    flat_target = flat_target[valid]

    conf  = torch.bincount(
        num_classes * flat_target + flat_pred,
        minlength=num_classes ** 2
    ).reshape(num_classes, num_classes).float()

    tp    = conf.diag()
    fp    = conf.sum(0) - tp
    fn    = conf.sum(1) - tp
    denom = tp + fp + fn
    return torch.where(denom > 0, tp / denom, torch.zeros_like(tp))


@torch.no_grad()
def mean_iou(pred: torch.Tensor, target: torch.Tensor, num_classes: int) -> float:
    return iou_per_class(pred, target, num_classes).mean().item()


@torch.no_grad()
def freq_weighted_iou(pred: torch.Tensor, target: torch.Tensor,
                      num_classes: int) -> float:
    """Frequency-weighted IoU: Σ_c  freq_c · IoU_c."""
    freq = torch.bincount(
        target.reshape(-1).clamp(0, num_classes - 1),
        minlength=num_classes
    ).float() / target.numel()
    iou = iou_per_class(pred, target, num_classes)
    return (freq * iou).sum().item()


def compute_metrics(pred, target, num_classes=NUM_CLASSES):
    return {
        'pixel_acc': pixel_accuracy(pred, target),
        'miou':      mean_iou(pred, target, num_classes),
        'fw_iou':    freq_weighted_iou(pred, target, num_classes),
    }

## 6. Training

**Setup**: Adam optimiser with cosine-annealing LR schedule. Each `run_training` call:
1. Trains for `config['epochs']` epochs
2. Evaluates on the validation set after every epoch
3. Logs scalar metrics to **wandb** (`train/loss`, `train/miou`, `val/loss`, `val/miou`, …)
4. Logs a grid of overlay images to wandb every 5 epochs (input · predicted mask blend · GT mask)

The `loss_fn` is selected from the config dict (`'ce'` / `'dice'` / `'combined'`) so the same loop can drive all three experiments.

In [ ]:
def _reset_wandb():
    try:
        subprocess.run(['pkill', '-f', 'wandb-core'], capture_output=True, timeout=5)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    time.sleep(0.5)
    os.environ.pop('WANDB_SERVICE', None)
    try:
        import wandb.sdk.wandb_setup as _ws
        _ws._WandbSetup._instance = None
    except Exception:
        pass


def _wandb_init(project, config, name=None, reset=True):
    logging.getLogger('wandb').setLevel(logging.ERROR)
    if reset:
        _reset_wandb()
    run = wandb.init(
        project=project,
        config=config,
        reinit=True,
        name=name,
        settings=wandb.Settings(silent=True),
    )
    wandb.define_metric('epoch')
    wandb.define_metric('*', step_metric='epoch')
    return run

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device, num_classes=NUM_CLASSES):
    model.train()
    total_loss = 0.0
    metric_sums = {'pixel_acc': 0.0, 'miou': 0.0, 'fw_iou': 0.0}
    n_batches = 0
    n_samples = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device).long()
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()
        bs   = imgs.size(0)
        pred = logits.argmax(dim=1)
        total_loss += loss.item() * bs
        for k, v in compute_metrics(pred, masks, num_classes).items():
            metric_sums[k] += v
        n_batches += 1
        n_samples += bs
    return {
        'loss': total_loss / n_samples,
        **{k: v / n_batches for k, v in metric_sums.items()},
    }


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, num_classes=NUM_CLASSES):
    model.eval()
    total_loss = 0.0
    metric_sums = {'pixel_acc': 0.0, 'miou': 0.0, 'fw_iou': 0.0}
    n_batches = 0
    n_samples = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device).long()
        logits = model(imgs)
        loss   = loss_fn(logits, masks)
        bs   = imgs.size(0)
        pred = logits.argmax(dim=1)
        total_loss += loss.item() * bs
        for k, v in compute_metrics(pred, masks, num_classes).items():
            metric_sums[k] += v
        n_batches += 1
        n_samples += bs
    return {
        'loss': total_loss / n_samples,
        **{k: v / n_batches for k, v in metric_sums.items()},
    }

In [ ]:
_PALETTE = np.array([[0, 0, 0], [200, 80, 80], [80, 80, 200]], dtype=np.uint8)


def log_overlay_images(model, loader, device, num_classes, step, n_images=4):
    """Log predicted mask overlay + GT mask to wandb."""
    model.eval()
    imgs, masks = next(iter(loader))
    imgs  = imgs[:n_images].to(device)
    masks = masks[:n_images].to(device)
    with torch.no_grad():
        preds = model(imgs).argmax(dim=1)
    wb_images = []
    for i in range(imgs.size(0)):
        img_np  = (imgs[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
        img_u8  = (img_np * 255).astype(np.uint8)
        pred_c  = _PALETTE[preds[i].cpu().numpy().clip(0, num_classes - 1)]
        gt_c    = _PALETTE[masks[i].cpu().numpy().clip(0, num_classes - 1)]
        overlay = (0.55 * img_u8 + 0.45 * pred_c).astype(np.uint8)
        wb_images += [
            wandb.Image(img_u8,  caption=f'[{i}] input'),
            wandb.Image(overlay, caption=f'[{i}] pred overlay'),
            wandb.Image(gt_c,    caption=f'[{i}] GT mask'),
        ]
    wandb.log({'visuals': wb_images, 'epoch': step})


def run_training(model, train_loader, val_loader, config, device,
                 num_classes=NUM_CLASSES, project='lab4-segmentation'):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config['lr'], weight_decay=config.get('wd', 1e-4))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['epochs'])

    loss_type = config.get('loss', 'ce')
    lam       = config.get('lam', 1.0)
    if loss_type == 'ce':
        loss_fn = lambda lo, ta: F.cross_entropy(lo, ta)
    elif loss_type == 'dice':
        loss_fn = lambda lo, ta: dice_loss(lo, ta, num_classes)
    else:
        loss_fn = lambda lo, ta: combined_loss(lo, ta, num_classes, lam)

    run = _wandb_init(project, config, name=config.get('name'))
    for epoch in range(1, config['epochs'] + 1):
        tr = train_one_epoch(model, train_loader, optimizer, loss_fn, device, num_classes)
        va = evaluate(model, val_loader, loss_fn, device, num_classes)
        scheduler.step()
        wandb.log({
            'epoch': epoch,
            **{f'train/{k}': v for k, v in tr.items()},
            **{f'val/{k}':   v for k, v in va.items()},
        })
        if epoch % 5 == 0 or epoch == config['epochs']:
            log_overlay_images(model, val_loader, device, num_classes, step=epoch)
        print(f'Epoch {epoch:3d} | '
              f'train loss={tr["loss"]:.4f} mIoU={tr["miou"]:.4f} | '
              f'val loss={va["loss"]:.4f} mIoU={va["miou"]:.4f}')
    run.finish()
    return model

## 7. Post-processing

Applied to **probability maps** after inference, before (or instead of) argmax:

| Technique | What it does |
|-----------|-------------|
| **Confidence threshold** | Pixels where `max_prob < τ` are forced to class 0 (background). Removes uncertain predictions. |
| **Probability smoothing** | Gaussian blur (`scipy.ndimage`) on each class probability map before argmax. Removes high-frequency noise. |
| **Blob removal** | Connected-component analysis (`cv2.connectedComponentsWithStats`) removes isolated regions smaller than `min_size` pixels. Eliminates stray detections. |

`evaluate_postprocessing` runs all four variants (baseline + the three above) on a loader in a single pass and returns averaged metrics for each.

In [ ]:
def apply_threshold(probs: torch.Tensor, threshold=0.5) -> torch.Tensor:
    """Pixels where max_prob < threshold are assigned class 0 (background)."""
    max_probs, pred = probs.max(dim=1)
    pred = pred.clone()
    pred[max_probs < threshold] = 0
    return pred


def smooth_probabilities(probs: torch.Tensor, sigma=1.0) -> torch.Tensor:
    """Gaussian blur on probability maps (spatial dims only) before argmax."""
    probs_np = probs.cpu().numpy()                                          # (B, C, H, W)
    smoothed = scipy.ndimage.gaussian_filter(probs_np, sigma=(0, 0, sigma, sigma))
    return torch.from_numpy(smoothed).to(probs.device)


def remove_small_blobs(mask: torch.Tensor, min_size=100) -> torch.Tensor:
    """Remove per-class connected components smaller than min_size pixels."""
    mask_np = mask.cpu().numpy()
    result  = np.zeros_like(mask_np)
    for b in range(mask_np.shape[0]):
        frame   = mask_np[b].astype(np.uint8)
        cleaned = frame.copy()
        if frame.max() == 0:
            result[b] = frame
            continue
        for c in range(1, int(frame.max()) + 1):
            binary                  = (frame == c).astype(np.uint8)
            n_lbl, labels, stats, _ = cv2.connectedComponentsWithStats(binary)
            for lbl in range(1, n_lbl):
                if stats[lbl, cv2.CC_STAT_AREA] < min_size:
                    cleaned[labels == lbl] = 0
        result[b] = cleaned
    return torch.from_numpy(result).long().to(mask.device)


@torch.no_grad()
def evaluate_postprocessing(model, loader, device, num_classes=NUM_CLASSES,
                             threshold=0.5, sigma=1.0, min_blob_size=100):
    """Evaluate baseline vs three post-processing variants."""
    model.eval()
    buckets = {k: [] for k in ('baseline', 'threshold', 'smooth', 'blob_removal')}
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device).long()
        probs = F.softmax(model(imgs), dim=1)

        buckets['baseline'].append(
            compute_metrics(probs.argmax(1), masks, num_classes))
        buckets['threshold'].append(
            compute_metrics(apply_threshold(probs, threshold), masks, num_classes))
        probs_s = smooth_probabilities(probs, sigma)
        buckets['smooth'].append(
            compute_metrics(probs_s.argmax(1), masks, num_classes))
        buckets['blob_removal'].append(
            compute_metrics(remove_small_blobs(probs.argmax(1), min_blob_size), masks, num_classes))

    return {
        name: {k: float(np.mean([m[k] for m in ms])) for k in ms[0]}
        for name, ms in buckets.items()
    }

## 8. Experiments

Three controlled comparisons, each with all other hyperparameters fixed:

| # | Variable | Options |
|---|----------|---------|
| 1 | Upsampling strategy | `ConvTranspose2d` vs bilinear `Upsample` + 1×1 conv |
| 2 | Training loss | CE · Dice · Combined (CE + Dice) |
| 3 | Post-processing | Baseline · threshold · smooth · blob removal |

**Fixed config**: combined loss (Exp 1), transposed-conv upsampling (Exp 2 & 3), enc_channels = (32, 64, 128, 256), 20 epochs, Adam lr = 3e-4.  
All runs are logged to wandb under the `lab4-segmentation` project.

### Setup — augmented DataLoaders

In [ ]:
BS = 8

train_ds_aug = LFWDataset(BASE_FOLDER,
                          transforms=make_train_transforms(IMG_SIZE),
                          split_name='train', download=False)
val_ds_aug   = LFWDataset(BASE_FOLDER,
                          transforms=make_val_transforms(IMG_SIZE),
                          split_name='val',   download=False)
test_ds_aug  = LFWDataset(BASE_FOLDER,
                          transforms=make_val_transforms(IMG_SIZE),
                          split_name='test',  download=False)

train_loader = DataLoader(train_ds_aug, batch_size=BS, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds_aug,   batch_size=BS, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds_aug,  batch_size=BS, shuffle=False, num_workers=0)

print(f'Batches — train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}')

EPOCHS = 20
LR     = 3e-4
ENC_CH = (32, 64, 128, 256)

### Experiment 1 — Upsampling strategy: transposed conv vs bilinear

**Hypothesis**: bilinear upsampling is smoother and less prone to checkerboard artefacts, but `ConvTranspose2d` can learn task-specific upsampling filters.  
Both models are trained with combined loss; results are compared on the held-out test set.

In [ ]:
exp1_results = {}
for use_bilinear, name in [(False, 'upsample-transposed'), (True, 'upsample-bilinear')]:
    print(f'\n=== {name} ===')
    model = UNet(in_channels=3, enc_channels=ENC_CH,
                 num_classes=NUM_CLASSES, use_bilinear=use_bilinear).to(DEVICE)
    cfg   = dict(loss='combined', lr=LR, epochs=EPOCHS,
                 use_bilinear=use_bilinear, name=name)
    model = run_training(model, train_loader, val_loader, cfg, DEVICE)
    exp1_results[name] = evaluate(model, test_loader,
                                   lambda lo, ta: combined_loss(lo, ta),
                                   DEVICE)

print('\n--- Experiment 1 results (test set) ---')
for name, m in exp1_results.items():
    print(f'{name:25s}  pixel_acc={m["pixel_acc"]:.4f}  mIoU={m["miou"]:.4f}  fw_iou={m["fw_iou"]:.4f}')

### Experiment 2 — Loss function: CE vs Dice vs combined

**Hypothesis**: CE provides strong per-pixel gradients early in training, while Dice targets overlap directly and is less sensitive to the background class dominating the loss. The combined loss is expected to benefit from both.

In [ ]:
exp2_results = {}
for loss_type in ('ce', 'dice', 'combined'):
    print(f'\n=== loss: {loss_type} ===')
    model = UNet(in_channels=3, enc_channels=ENC_CH,
                 num_classes=NUM_CLASSES, use_bilinear=False).to(DEVICE)
    cfg   = dict(loss=loss_type, lr=LR, epochs=EPOCHS, name=f'loss-{loss_type}')
    model = run_training(model, train_loader, val_loader, cfg, DEVICE)
    exp2_results[loss_type] = evaluate(model, test_loader,
                                        lambda lo, ta: combined_loss(lo, ta),
                                        DEVICE)

print('\n--- Experiment 2 results (test set) ---')
for lt, m in exp2_results.items():
    print(f'{lt:10s}  pixel_acc={m["pixel_acc"]:.4f}  mIoU={m["miou"]:.4f}  fw_iou={m["fw_iou"]:.4f}')

### Experiment 3 — Post-processing impact

A single reference model (combined loss, transposed-conv upsampling) is trained once, then all four post-processing variants are evaluated on the **test set** in a single `evaluate_postprocessing` pass.  
Results are printed in a table and logged to wandb for side-by-side comparison.

In [ ]:
# Train a reference model (combined loss, transposed-conv upsampling)
print('Training reference model ...')
ref_model = UNet(in_channels=3, enc_channels=ENC_CH,
                 num_classes=NUM_CLASSES, use_bilinear=False).to(DEVICE)
cfg = dict(loss='combined', lr=LR, epochs=EPOCHS, name='postproc-reference')
ref_model = run_training(ref_model, train_loader, val_loader, cfg, DEVICE)

In [ ]:
pp_results = evaluate_postprocessing(
    ref_model, test_loader, DEVICE,
    threshold=0.5, sigma=1.0, min_blob_size=100)

print('\n--- Post-processing results (test set) ---')
for variant, m in pp_results.items():
    print(f'{variant:15s}  pixel_acc={m["pixel_acc"]:.4f}  '
          f'mIoU={m["miou"]:.4f}  fw_iou={m["fw_iou"]:.4f}')

# Log to wandb
run = _wandb_init('lab4-segmentation', config={'experiment': 'postprocessing'},
                  name='postproc-comparison')
for variant, m in pp_results.items():
    wandb.log({'epoch': 0, **{f'postproc/{variant}/{k}': v for k, v in m.items()}})
run.finish()

In [ ]:
# Visualize post-processing on a few test examples
ref_model.eval()
imgs_vis, masks_vis = next(iter(test_loader))
imgs_vis  = imgs_vis[:2].to(DEVICE)
masks_vis = masks_vis[:2]

with torch.no_grad():
    probs_vis = F.softmax(ref_model(imgs_vis), dim=1)

variants = {
    'Baseline':      probs_vis.argmax(1).cpu(),
    'Threshold':     apply_threshold(probs_vis, 0.5).cpu(),
    'Smooth':        smooth_probabilities(probs_vis, sigma=1.0).argmax(1).cpu(),
    'BlobRemoval':   remove_small_blobs(probs_vis.argmax(1), min_size=100).cpu(),
    'GT':            masks_vis,
}

n_variants = len(variants)
fig, axes = plt.subplots(2, n_variants + 1, figsize=(3 * (n_variants + 1), 6))
for row in range(2):
    img_np = (imgs_vis[row].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Input')
    axes[row, 0].axis('off')
    for col, (name, preds) in enumerate(variants.items(), start=1):
        axes[row, col].imshow(preds[row].numpy(), cmap=cmap, vmin=0, vmax=2)
        axes[row, col].set_title(name)
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()